## Phase 1: Advanced Dependency Initialization
Installing the heavy-duty dependencies required for Slicing Aided Hyper Inference (SAHI), Zero-Shot Segmentation (MobileSAM), and Web UI Deployment (Streamlit). 


In [4]:


import torch
import gc

# 2. VRAM Safety Flush
# Deep Learning golden rule: Always sweep dead memory before starting a complex pipeline
torch.cuda.empty_cache()
gc.collect()

# 3. Hardware Lock
device_str = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device = torch.device(device_str)

print(f"✅ Sentinel-AI Environment Initialized.")
print(f"🔒 Hardware locked to: {device}")


✅ Sentinel-AI Environment Initialized.
🔒 Hardware locked to: cuda:0


## Phase 2: High-Resolution Coordinate Extraction (SAHI)
Executing Slicing Aided Hyper Inference to overcome resolution loss. We extract the raw bounding box coordinate arrays `[x_min, y_min, x_max, y_max]` to act as geometric prompts for the segmentation engine.


In [5]:
import os
import urllib.request
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# 1. Download a stable test image 
test_image_path = "sentinel_test.jpg"
if not os.path.exists(test_image_path):
    print("📥 Downloading test image...")
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", test_image_path)

# 2. Load the Detection Brain (YOLO via SAHI)
print("🧠 Loading YOLOv8 into SAHI Engine...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path='visdrone_15epochs_best.pt', 
    confidence_threshold=0.30,
    device="cuda:0"
)

# 3. Mathematical Slicing & Detection
print("🔪 Executing Sliced Inference...")
result = get_sliced_prediction(
    test_image_path,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2
)

# 4. Extract Bounding Box Prompts for Phase 3
sahi_bboxes = [obj.bbox.to_xyxy() for obj in result.object_prediction_list]
print(f"🎯 Successfully extracted {len(sahi_bboxes)} bounding box prompts for SAM!")


📥 Downloading test image...
🧠 Loading YOLOv8 into SAHI Engine...
🔪 Executing Sliced Inference...
Performing prediction on 4 slices.
🎯 Successfully extracted 2 bounding box prompts for SAM!


## Phase 3: Zero-Shot Segmentation (MobileSAM)
Executing VRAM Offloading to safely load the Segment Anything Model. We utilize the SAHI coordinate arrays as geometric prompts to generate pixel-perfect object masks.


In [6]:
from ultralytics import SAM
import torch
import gc

# 1. VRAM Juggling: Evict YOLO from the GPU
print("🧹 Evicting YOLO from VRAM...")
detection_model.model.to('cpu') # Push YOLO back to standard laptop RAM
torch.cuda.empty_cache()
gc.collect()

# 2. Load MobileSAM into the newly cleared VRAM
print("🧠 Loading MobileSAM into VRAM...")
sam_model = SAM('mobile_sam.pt')

# 3. Geometric Prompting
print("✨ Prompting SAM with SAHI coordinates...")

if len(sahi_bboxes) > 0:
    # We pass the bounding boxes we extracted in Phase 2 directly to SAM
    sam_results = sam_model.predict(
        test_image_path, 
        bboxes=sahi_bboxes, 
        save=True, 
        device="cuda:0"
    )
    print("✅ Segmentation Complete! Open the 'runs/segment/predict' folder to see the masks!")
else:
    print("❌ No bounding boxes found to prompt SAM with.")


🧹 Evicting YOLO from VRAM...
🧠 Loading MobileSAM into VRAM...
✨ Prompting SAM with SAHI coordinates...

image 1/1 d:\Uttam\Project\Own_Project\Deep Learning\001\SENTINEL_AI\sentinel_test.jpg: 1024x1024 1 0, 1 1, 396.2ms
Speed: 15.9ms preprocess, 396.2ms inference, 45.2ms postprocess per image at shape (1, 3, 1024, 1024)
Results saved to D:\Uttam\Project\Own_Project\Deep Learning\001\SENTINEL_AI\runs\segment\predict
✅ Segmentation Complete! Open the 'runs/segment/predict' folder to see the masks!


## Phase 4: Streamlit Web Dashboard
Deploying the custom VisDrone model into an interactive Web UI. We utilize `@st.cache_resource` to lock the model in VRAM, preventing memory crashes when the user interacts with the live sliders.


In [7]:
%%writefile app.py
import streamlit as st
from ultralytics import YOLO
from PIL import Image

st.set_page_config(page_title="Sentinel-AI", layout="wide")
st.title("🚁 Sentinel-AI: Mass Drone Vision")

@st.cache_resource
def load_model():
    return YOLO('visdrone_15epochs_best.pt')

model = load_model()

st.sidebar.header("Radar Settings")
conf_thresh = st.sidebar.slider("Confidence Threshold", 0.0, 1.0, 0.25)

# Enable multiple files!
uploaded_files = st.file_uploader("Upload 25+ Drone Images...", type=["jpg", "png", "jpeg"], accept_multiple_files=True)

if uploaded_files:
    st.write(f"🧠 Processing {len(uploaded_files)} images...")
    
    # Loop through all 25 images
    for file in uploaded_files:
        image = Image.open(file)
        results = model.predict(source=image, conf=conf_thresh)
        rendered_image = results[0].plot() 
        
        st.image(rendered_image, caption=f"Detection: {file.name}", use_column_width=True)


Writing app.py
